# Credit Card Fraud Detection - PCA Encrypted Dataset

**Dataset Overview & Anomaly Detection:**
This dataset contains credit card transactions, and the primary goal is anomaly detection (identifying fraudulent transactions among legitimate ones).
Due to confidentiality and privacy concerns, the dataset is a **PCA (Principal Component Analysis) encrypted dataset**. 
- The original features are not provided.
- Features `V1`, `V2`, ... `V28` are the principal components obtained through the PCA transformation.
- The only features that have not been transformed are `Time` (seconds elapsed between each transaction and the first transaction) and `Amount` (transaction amount).
- The `Class` feature is the target variable, where `1` represents a fraudulent transaction and `0` represents a legitimate transaction.

This severe class imbalance requires specialized handling and evaluation metrics.

### Importing Required Libraries
We import libraries for data analysis (`pandas`, `numpy`), visualization (`matplotlib`, `seaborn`), and machine learning preprocessing and modeling (`sklearn`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

### Loading the Dataset
We load the transaction data from `creditcard.csv` into a pandas DataFrame.

In [ ]:
# load the data set

df = pd.read_csv("creditcard.csv")

### Inspecting the Data
We use `.head()` to preview the first five rows of the dataset.

In [ ]:
# print the basic info

df.head()

### Dataset Dimensions
We check the `.shape` of the DataFrame to see the number of rows and columns.

In [ ]:
# shape of the data set

df.shape

### Dataset Information
We use `.info()` to get a summary of the dataset's column data types and non-null values.

In [ ]:
# info of the data set

df.info()

### Statistical Summary
We use `.describe()` to view the basic statistical details like mean, standard deviation, and percentiles for the numerical features.

In [ ]:
# description of the data set

df.describe()

### Missing Value Checker
We verify if there are any missing values in the dataset using `.isnull().sum()`.

In [ ]:
# null value checker

df.isnull().sum()

### Class Distribution Math
We calculate the absolute counts and the percentage of legitimate vs. fraudulent transactions to quantify the class imbalance.

In [ ]:
# Class distribution math
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print("\n--- Class distribution ---")
print(class_counts)
print("\n--- Class distribution (%) ---")
print(class_pct.round(4))

### The Visual Proof (Seaborn)
We visualize the class distribution using a count plot. A logarithmic scale is applied to the y-axis so that the very small number of fraudulent transactions is visible.

In [ ]:
# The Visual Proof (Seaborn)
plt.figure(figsize=(5, 4))
sns.countplot(x='Class', data=df)
plt.title("Class Distribution: 0 = Genuine, 1 = Fraud")
plt.yscale('log')  # Log scale makes the tiny 0.17% fraud bar visible
plt.savefig("graphs_of_dataset_statistics/class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

### Feature Correlation Analysis: Full Heatmap

To begin the feature selection process, we first need to understand how the features relate to one another and to our target variable (`Class`). 
We calculate the Pearson correlation coefficient matrix for the entire dataset. A heatmap is then generated to visualize these correlations. 
- **High positive or negative correlations between independent features** (multicollinearity) might indicate redundant information. 
- **Correlations with the target variable** help us identify which features might be the most predictive. 
*Note: We save this visualization as `full_correlation_heatmap.png` for reporting.*

In [ ]:
plt.figure(figsize=(20, 14))
corr_matrix = df.corr()

# Seaborn paints the heatmap
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=True, fmt='.2f', annot_kws={'size': 8})
plt.title("Correlation Heatmap - All Features")
plt.savefig("graphs_of_dataset_statistics/full_correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

### Selecting Features via Pearson Correlation to Target

While the full heatmap gives a macroscopic view, we are most interested in how each feature correlates specifically with the `Class` (fraud or genuine). 
In this step, we:
1. Extract the correlation values of all features against the `Class` column.
2. Sort them by their **absolute magnitude** to identify the strongest linear relationships, whether positive or negative.
3. Visualize these linear relationships using a horizontal bar chart (green for positive correlation, red for negative).
4. Extract the **top 10 most correlated features**. This provides us with our first candidate subset of predictive features.

In [ ]:
# Grab absolute correlations to the target, sort highest to lowest
target_corr = corr_matrix['Class'].drop('Class').sort_values(key=abs, ascending=False)

plt.figure(figsize=(8, 10))
target_corr.plot(kind='barh', color=['red' if v < 0 else 'green' for v in target_corr])
plt.title("Feature Correlation with Class (Fraud)")
plt.xlabel("Pearson Correlation Coefficient")
plt.gca().invert_yaxis()
plt.savefig("graphs_of_dataset_statistics/feature_target_correlation.png", dpi=150, bbox_inches='tight')
plt.show()

# Save the top 10 features into a list
N = 10
top_features = target_corr.abs().sort_values(ascending=False).head(N).index.tolist()
print(f"\n--- Top {N} features selected by Pearson correlation ---")
print(top_features)

### Tree-Based Feature Selection: Random Forest Importance

Linear correlation (Pearson) only captures linear relationships. To capture complex, non-linear relationships and interactions between features, we utilize a tree-based model.
Here, we:
1. Separate our features (`X`) and target (`y`).
2. **Scale the Data:** We apply `StandardScaler` to the `Amount` and `Time` columns. While Random Forests are scale-invariant, scaling is a best practice if we plan to use these features in distance-based models later. *(Note: The PCA features V1-V28 are already scaled).* 
3. **Train a Random Forest:** We train a `RandomForestClassifier`. Crucially, we set `class_weight='balanced'` to penalize mistakes on the minority fraud class, preventing the model from just predicting 'genuine' every time.
4. **Extract Importances:** We extract the `feature_importances_` attribute, which measures how much each feature contributes to reducing impurity across all trees in the forest.
5. We visualize and extract the **top 10 features** identified by the Random Forest.

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

# Scale Amount and Time
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[['Amount', 'Time']] = scaler.fit_transform(X[['Amount', 'Time']])

# Train the Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1 # This tells your i7 processor to use all its cores!
)
rf.fit(X_scaled, y)

# Extract and plot the best features
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 10))
importances.head(N).plot(kind='barh', color='steelblue')
plt.title(f"Top {N} Feature Importances (Random Forest)")
#plt.gca().invert_yaxis()
#plt.savefig("graphs_of_dataset_statistics/rf_feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

### Final Feature Set: The Magic Merge (Union)

We now have two distinct perspectives on feature importance:
1. **Pearson Correlation:** Identifies strong linear relationships.
2. **Random Forest:** Identifies non-linear patterns and complex feature interactions.

To build the most robust predictive model, we combine the insights from both methods. By performing a mathematical **Union** of the two sets of top 10 features, we ensure that we do not miss out on features that might be deemed important by one method but overlooked by the other. This final merged set will be used for training our subsequent anomaly detection classification models.

In [ ]:
# Grab the top 10 from Random Forest
rf_top = importances.head(N).index.tolist()

# The Magic Merge (Union)
final_features = sorted(set(top_features) | set(rf_top))

print(f"\n--- FINAL SELECTED FEATURES ({len(final_features)}) ---")
print(final_features)